[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# ¿Qué es un agente? El ciclo, la autonomía y los modos de falla.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 1: *Fundamentos de sistemas agénticos*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se definirá qué es un agente, se construirá **a mano** el ciclo de razonar, llamar herramientas y observar, y se revisarán los patrones de diseño y los modos de falla más comunes. Todo lo que hace `create_agent` en los siguientes *notebooks* es una versión robusta de este ciclo.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del curso): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del curso (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue el *notebook* `00_claves_de_api_y_secretos.ipynb`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este curso usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.5-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue el notebook 00_claves_de_api_y_secretos: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
import json

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

---
## 1. Del modelo al agente.

Un **agente** es un sistema en el que un modelo de lenguaje decide, en un ciclo, qué acción tomar a continuación:

**agente = modelo + herramientas + ciclo de decisión + estado.**

| Nivel de autonomía | Quién decide el siguiente paso | Ejemplo |
|---|---|---|
| Cadena fija | El código, siempre igual | Resumir un correo |
| *Router* | El modelo elige **una** rama entre varias predefinidas | Clasificar una solicitud y mandarla a un área |
| Agente con herramientas | El modelo decide qué herramienta usar y cuándo terminar | Asistente que consulta políticas y abre tickets |
| Agente profundo | El modelo además planifica, escribe archivos y delega | Investigación de mercado de varias horas |
| Sistema multiagente | Varios agentes se reparten el trabajo | Supervisor con agentes especialistas |

**Regla práctica:** usa el nivel más bajo que resuelva el problema. Cada nivel agrega flexibilidad, pero también costo, latencia y formas nuevas de fallar.

---
## 2. El ciclo del agente, a mano.

El ciclo tiene tres pasos que se repiten:
1. **Razonar:** el modelo recibe los mensajes y responde con texto **o** con una o más *tool calls*.
2. **Actuar:** el código ejecuta cada herramienta pedida.
3. **Observar:** el resultado vuelve al modelo como `ToolMessage` y el ciclo se repite hasta que el modelo responde sin pedir herramientas.

* La siguiente celda definirá dos herramientas para consultar las políticas de *Andina Logística*.

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda implementará el ciclo completo en una función de unas 15 líneas.

In [ ]:
herramientas = {h.name: h for h in [listar_politicas, leer_politica]}
modelo = init_chat_model(MODELO).bind_tools(list(herramientas.values()))


def ciclo_agente(pregunta: str, max_pasos: int = 5):
    mensajes = [SystemMessage("Responde solo con base en las políticas internas."), HumanMessage(pregunta)]
    for paso in range(1, max_pasos + 1):
        respuesta = modelo.invoke(mensajes)                     # 1. razonar
        mensajes.append(respuesta)
        if not respuesta.tool_calls:                             # sin herramientas: terminó
            return respuesta.text, mensajes
        for llamada in respuesta.tool_calls:                    # 2. actuar
            print(f"paso {paso}: {llamada['name']}({json.dumps(llamada['args'], ensure_ascii=False)})")
            resultado = herramientas[llamada["name"]].invoke(llamada["args"])
            mensajes.append(ToolMessage(str(resultado), tool_call_id=llamada["id"]))  # 3. observar
    return "Se alcanzó el límite de pasos.", mensajes

* La siguiente celda ejecutará el ciclo con una pregunta sobre vacaciones.

In [ ]:
texto, mensajes = ciclo_agente("¿Cuántos días de vacaciones me tocan en mi tercer año?")
texto

**Observa:** el modelo primero lista las políticas, luego lee `vacaciones` y solo entonces responde. Nadie le dijo ese orden: lo decidió a partir de las descripciones de las herramientas.

**💡 Ideas clave de la sección 2:**
* El modelo **no ejecuta** herramientas: devuelve *tool calls* y el código las ejecuta.
* El estado del agente es la lista de mensajes; crece en cada paso.
* Sin un límite de pasos, un agente puede quedar en un bucle y gastar tokens sin fin.

### Ejercicio 1 · Contar pasos y tokens.

Modifica `ciclo_agente` para que, además del texto, devuelva el número de llamadas al modelo y la suma de tokens de entrada y salida. Pista: cada `AIMessage` tiene el atributo `usage_metadata` con `input_tokens` y `output_tokens`. Pruébalo con la pregunta anterior.

In [ ]:
# TU CÓDIGO AQUÍ

---
## 3. Patrones de diseño.

| Patrón | Idea | Cuándo usarlo |
|---|---|---|
| **ReAct** | Alternar razonamiento y acción en un ciclo (lo que se construyó arriba) | Tareas cortas con herramientas |
| **Planificar y ejecutar** | Primero un plan completo, después ejecutar cada paso | Tareas largas con pasos predecibles |
| **Reflexión** | El modelo revisa y corrige su propia respuesta antes de entregarla | Cuando la calidad importa más que la latencia |
| **Orquestador y trabajadores** | Un agente reparte subtareas a otros especializados | Problemas que se dividen por dominio |

---
## 4. Modos de falla.

| Falla | Síntoma | Defensa (*notebook*) |
|---|---|---|
| Bucle | Llama la misma herramienta una y otra vez | Límites de llamadas (`08`) |
| Herramienta inventada | Pide una herramienta o argumento que no existe | Esquemas estrictos y errores útiles (`05`) |
| Deriva del objetivo | Termina haciendo otra cosa | *Prompt* claro, evaluación (`13`) |
| Costo sin límite | Historial enorme, muchos pasos | Resumen del historial, límites (`08`, `14`) |
| Inyección de instrucciones | Un documento o resultado de herramienta le da órdenes | Permisos mínimos, aprobación humana (`08`, `14`) |

### Ejercicio 2 · Caso de diseño: ¿flujo o agente?

Para cada caso, decide qué nivel de autonomía usarías (tabla de la sección 1) y justifica en una línea:
1. Clasificar cada correo entrante de soporte en una de 5 categorías.
2. Revisar un contrato de 40 páginas y señalar cláusulas de riesgo.
3. Conciliar mensualmente 2,000 movimientos bancarios contra facturas.

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Reflexión.

Agrega a `ciclo_agente` un paso final de **reflexión**: antes de devolver la respuesta, pide al modelo (sin herramientas) que verifique si la respuesta está respaldada por las políticas leídas y que la corrija si no.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar el ejercicio 1 y comparar en *LangSmith* las trazas de `ciclo_agente` con distintas preguntas.

**Siguiente *notebook*:** `03_ecosistema_langchain_v1.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>